# Model Testing

# Work on whole class video

In [ ]:
!pip install ultralytics opencv-python torch torchvision

In [ ]:
import cv2
import os
import torch
import numpy as np
from ultralytics import YOLO
from torchvision.models.video import r3d_18


In [ ]:
# Load Models
device = "cuda" if torch.cuda.is_available() else "cpu"

# YOLO for student detection + tracking
yolo = YOLO("yolov8n.pt")

# R3D‑18 for cheating classification
model = r3d_18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, 5)
model.load_state_dict(torch.load("/content/drive/MyDrive/My_Models/r3d18_cheating_194_batch-4.pth"))
model = model.to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


VideoResNet(
  (stem): BasicStem(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1):

In [ ]:
# classes
class_names = [
    "copying",
    "gesture_share",
    "mobile_use",
    "notes_use",
    "normal"
]


In [ ]:
# step no 4: Frame --> Clip Function (critcal part)
def make_clip(frames):
    clip = []
    for f in frames:
        f = cv2.resize(f, (112,112))
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        f = torch.from_numpy(f).permute(2,0,1).float() / 255.0
        clip.append(f)

    clip = torch.stack(clip)           # [T,C,H,W]
    clip = clip.permute(1,0,2,3)        # [C,T,H,W]
    return clip.unsqueeze(0).to(device)


In [ ]:
# step 5: Predict clip
def predict_clip(clip):
    with torch.no_grad():
        out = model(clip)
        prob = torch.softmax(out, dim=1)[0]
        cls = torch.argmax(prob).item()
    return class_names[cls], float(prob[cls])


In [ ]:
# Upload whole class video
from google.colab import files

# print("Upload WHOLE CLASS video:")
# uploaded = files.upload()

# video_name = list(uploaded.keys())[0]

video_name = "/content/DSC_0245.MOV"
# logs = analyze_class_video(video_name)


In [ ]:
# Step 6: Process whole class video; log too long so log save in chunks form
import pandas as pd
def analyze_class_video(video_path, save_csv_path, chunk_save_interval=200):

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    student_buffers = {}
    frame_no = 0
    CLIP_LEN = 16

    tracker_to_student = {}
    student_counter = 0

    # Create CSV file with header first
    if not os.path.exists(save_csv_path):
        pd.DataFrame(columns=[
            "student_id",
            "start_time_s",
            "end_time_s",
            "cheating_type",
            "confidence"
        ]).to_csv(save_csv_path, index=False)


    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            # for box, sid in zip(boxes, ids):
            for box, track_id in zip(boxes, ids):
                if track_id not in tracker_to_student:
                     student_counter += 1
                     tracker_to_student[track_id] = student_counter
                sid = tracker_to_student[track_id]


                x1,y1,x2,y2 = map(int, box)
                crop = frame[y1:y2, x1:x2]

                if sid not in student_buffers:
                    student_buffers[sid] = []

                student_buffers[sid].append(crop)

                if len(student_buffers[sid]) == CLIP_LEN:
                    clip = make_clip(student_buffers[sid])
                    label, conf = predict_clip(clip)

                    start_t = frame_no / fps
                    end_t = (frame_no + CLIP_LEN) / fps

                    row = {
                        "student_id": sid,
                        "start_time_s": round(start_t,2),
                        "end_time_s": round(end_t,2),
                        "cheating_type": label,
                        "confidence": round(conf,2)
                    }

                    # 🔥 Append immediately to CSV
                    pd.DataFrame([row]).to_csv(
                        save_csv_path,
                        mode='a',
                        header=False,
                        index=False
                    )

                    student_buffers[sid].pop(0)

        # 🔥 Save progress info every N frames
        if frame_no % chunk_save_interval == 0:
            print(f"Processed frame: {frame_no}")

        frame_no += 1

    cap.release()
    print("✅ Finished processing.")

In [ ]:
# Log of step 6: chunk file run
csv_path = "/content/drive/MyDrive/My_Models/DSC_0122_log.csv"
analyze_class_video(video_name, csv_path)

NameError: name 'analyze_class_video' is not defined

In [ ]:
# Step 8: Review Log from csv
import pandas as pd

csv_path = "/content/drive/MyDrive/My_Models/DSC_0245_log.csv"

log_df = pd.read_csv(csv_path)

print("===== Cheating Log =====")
print(log_df.head())  # show first rows

===== Cheating Log =====
   student_id  start_time_s  end_time_s cheating_type  confidence
0           1          0.25        0.52       copying        0.45
1           2          0.25        0.52     notes_use        0.60
2           3          0.25        0.52     notes_use        0.90
3           4          0.25        0.52       copying        0.45
4           5          0.25        0.52     notes_use        0.61


In [ ]:
# preview video
from IPython.display import HTML
from base64 import b64encode
import pandas as pd
import cv2

annotated_video_path = "/content/annotated_DSC_0122.mp4"

# Load log CSV
log_df = pd.read_csv(csv_path)

cap = cv2.VideoCapture(video_name)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(annotated_video_path, fourcc, fps, (width, height))

# 🔥 TRACKER → STUDENT MAPPING
tracker_to_student = {}
student_counter = 0

frame_no = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):

            # Stable student ID
            if track_id not in tracker_to_student:
                student_counter += 1
                tracker_to_student[track_id] = student_counter

            sid = tracker_to_student[track_id]

            x1,y1,x2,y2 = map(int, box)

            current_time = frame_no / fps

            # Find log row for this time
            student_events = log_df[
                (log_df["student_id"] == sid) &
                (log_df["start_time_s"] <= current_time) &
                (log_df["end_time_s"] >= current_time)
            ]

            # Default
            label = "normal"
            conf = 0.0
            box_color = (0,255,0)  # GREEN

            if len(student_events) > 0:
                label = student_events.iloc[-1]["cheating_type"]
                conf = student_events.iloc[-1]["confidence"]

                if label != "normal":
                    box_color = (0,0,255)  # RED

            # Draw bounding box
            cv2.rectangle(frame, (x1,y1), (x2,y2), box_color, 3)

            # Prepare text
            text = f"SID:{sid} {label} ({conf:.2f})"

            # Text background (NAVY BLUE)
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(frame, (x1, y1-35), (x1+tw+10, y1), (128,0,0), -1)

            # White text
            cv2.putText(frame, text, (x1+5, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (255,255,255), 2)

    out.write(frame)
    frame_no += 1

cap.release()
out.release()

print("✅ Annotated video saved.")

# Display
mp4 = open(annotated_video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=800 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")